**Data Ingestion**

1) Imports



In [ ]:
import os, time, pickle
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import NumericType
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier, DecisionTreeClassifier,
    GBTClassifier, OneVsRest
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from pyspark.mllib.evaluation import MulticlassMetrics


2) Spark Session + Config

In [ ]:
# 2) Spark Session + Config (Colab + Spark UI + Event Logs)
import os, shutil
from pyspark.sql import SparkSession

# Where Spark will write event logs (used by History Server / evidence)
EVENT_DIR = "/content/spark-events"
if os.path.exists(EVENT_DIR):
    shutil.rmtree(EVENT_DIR)
os.makedirs(EVENT_DIR, exist_ok=True)

# Where Spark checkpointing will write
CHECKPOINT_DIR = "/content/spark-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("OpenFoodFacts_NOVA_Classification")
    .master("local[*]")  # enables parallelism across CPU cores
    # memory / stability
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "512m")
    # performance tuning
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.files.maxPartitionBytes", str(32 * 1024 * 1024))  # 32MB
    # Spark UI
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    # Event Logging
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", f"file:{EVENT_DIR}")
    .config("spark.eventLog.compress", "false")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")
sc.setCheckpointDir(CHECKPOINT_DIR)

print("Spark version:", spark.version)
print("Spark UI internal:", spark.sparkContext.uiWebUrl)
print("Event log dir:", EVENT_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

 3) Load Data (raw_df)


In [ ]:
from huggingface_hub import hf_hub_download
from pyspark.sql import functions as F

# Load Open Food Facts parquet (big dataset)
path = hf_hub_download(
    repo_id="openfoodfacts/product-database",
    filename="food.parquet",
    repo_type="dataset"
)

needed_cols = [
    "nova_group", "code", "additives_n", "ingredients_n", "nutriments",
    "product_name", "brands", "categories", "countries", "packaging", "labels"
]

_base = spark.read.parquet(path)
raw_df = _base.select([c for c in needed_cols if c in _base.columns])

# Early filter (lazy; reduces later work)
raw_df = raw_df.filter(F.col("nova_group").isNotNull())

print("Loaded columns:", raw_df.columns)
raw_df.printSchema()
raw_df.limit(5).show(truncate=False)

# Open Spark UI after first action
from google.colab import output
import re

ui_url = spark.sparkContext.uiWebUrl
print("Spark UI internal URL:", ui_url)

m = re.search(r":(\d+)", ui_url or "")
port = int(m.group(1)) if m else 4040

print("Opening Spark UI on port:", port)
output.serve_kernel_port_as_iframe(port)


4) Ingestion Validation (raw stage)

In [ ]:
from pyspark.sql import functions as F
import os
import pandas as pd

print("=== Ingestion validation (Colab-safe) ===")

# Big-data evidence: file size (>=1GB requirement)
size_bytes = os.path.getsize(path)
print("Downloaded parquet size (GB):", round(size_bytes / (1024**3), 3))

print("Columns loaded:", raw_df.columns)

# Safe preview (no full scans)
raw_df.select("nova_group", "code", "additives_n", "ingredients_n").limit(20).show(truncate=False)

# Safe NOVA distribution preview using Python-side counts (avoids Spark shuffle on huge raw_df)
nova_sample = raw_df.select("nova_group").limit(2000).toPandas()
print("Preview NOVA distribution (first 2000 rows):")
print(nova_sample["nova_group"].value_counts(dropna=False).sort_index())


5) Working Dataset Creation

In [ ]:
from pyspark.sql import functions as F
from pyspark import StorageLevel

# Minimal columns required for ML pipeline
raw_min = raw_df.select("nova_group", "code", "additives_n", "ingredients_n", "nutriments")

# Start small and increase if stable (0.01 → 0.05 → 0.10)
WORK_FRAC = 0.02

# avoid counting or persisting while nutrients are still present since this may cause OOM.
work_df = raw_min.sample(withReplacement=False, fraction=WORK_FRAC, seed=42)

# Safe preview (without a complete scan)
work_df.limit(5).show(truncate=False)
print("WORK_FRAC =", WORK_FRAC)
print("Since, nutriments column is very large, so let's skip work_df.count() to avoid OOM.")

# ---- Feature extraction: nutriments is array<struct<name,value,...>> ----
# Safe extractor: returns NULL if nutriment not found
def nutr_value_safe(nutr_name: str):
    arr = F.expr(f"filter(nutriments, x -> x.name = '{nutr_name}')")
    return F.when(
        F.size(arr) > 0,
        F.element_at(arr, 1).value.cast("double")
    ).otherwise(F.lit(None).cast("double"))

# using names i.e. removing _100g that comes under "nutriments"
nutri_features = {
    "energy_100g": "energy",
    "fat_100g": "fat",
    "sugars_100g": "sugars",
    "salt_100g": "salt",
    "proteins_100g": "proteins",
    "carbohydrates_100g": "carbohydrates"
}

df = work_df.withColumn("nova_group", F.col("nova_group").cast("int"))

for out_col, nutr_name in nutri_features.items():
    df = df.withColumn(out_col, nutr_value_safe(nutr_name))

# After extraction, fill missing numeric values
fill_map = {c: 0.0 for c in nutri_features.keys()}
fill_map.update({"additives_n": 0.0, "ingredients_n": 0.0})
df = df.fillna(fill_map)

# Remove invalid negatives
for c in nutri_features.keys():
    df = df.filter(F.col(c) >= 0)

# Drop nutriments (big column) before caching
optimized_df = df.select(
    "nova_group", "code", "additives_n", "ingredients_n",
    *list(nutri_features.keys())
)

# now it is safe to persist + count
optimized_df = optimized_df.persist(StorageLevel.MEMORY_AND_DISK)

optimized_df.limit(5).show(truncate=False)
print("optimized_df columns:", optimized_df.columns)
print("optimized_df rows:", optimized_df.count())